In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from fileformer import FileFormer
from torchinfo import summary

In [2]:
model = FileFormer(257, 256, 128, 4, 6, 0.0)

In [3]:
summary(model)

Layer (type:depth-idx)                   Param #
FileFormer                               --
├─Embedding: 1-1                         65,792
├─LearnablePositionalEmbeddings: 1-2     --
│    └─Embedding: 2-1                    32,768
├─ModuleList: 1-3                        --
│    └─FileFormerBlock: 2-2              --
│    │    └─MultiHeadAttention: 3-1      262,400
│    │    └─FeedForward: 3-2             525,568
│    │    └─LayerNorm: 3-3               512
│    │    └─LayerNorm: 3-4               512
│    │    └─Dropout: 3-5                 --
│    └─FileFormerBlock: 2-3              --
│    │    └─MultiHeadAttention: 3-6      262,400
│    │    └─FeedForward: 3-7             525,568
│    │    └─LayerNorm: 3-8               512
│    │    └─LayerNorm: 3-9               512
│    │    └─Dropout: 3-10                --
│    └─FileFormerBlock: 2-4              --
│    │    └─MultiHeadAttention: 3-11     262,400
│    │    └─FeedForward: 3-12            525,568
│    │    └─LayerNorm: 3-13  

In [8]:
x = torch.randint(1, 100, (2, 128), dtype=torch.long)
msk = torch.zeros(2, 128, dtype=torch.bool)

In [12]:
out = model(x, msk)

In [11]:
model = model.to(torch.bfloat16)

In [13]:
out

tensor([[[ 0.8242, -1.8125, -0.5117,  ...,  0.2500, -0.9648, -0.7031],
         [ 1.1797, -1.2188, -0.6719,  ..., -0.7031,  0.7227,  0.5078],
         [-0.0173, -0.2539,  1.0391,  ..., -0.2158, -1.3281,  0.3398],
         ...,
         [ 1.2969,  0.5742, -1.2500,  ..., -1.0156, -0.1748, -1.8906],
         [ 0.6289,  0.9453,  0.5156,  ..., -1.8750, -0.2354, -0.5977],
         [ 0.1953, -0.5742,  1.2812,  ...,  0.2812, -1.0391, -1.5547]],

        [[-0.2275, -3.4375, -1.3203,  ..., -0.4043, -1.6719,  0.2090],
         [ 1.1406, -0.2559,  0.0435,  ..., -0.0259,  1.9219,  1.1641],
         [ 0.2715, -0.5938,  0.9570,  ..., -0.8594,  0.3926, -0.1475],
         ...,
         [ 1.4453,  0.6367,  0.2236,  ...,  0.4844, -0.4824, -0.3711],
         [ 1.9062, -0.1943,  1.1719,  ..., -0.4961, -1.3594, -1.1953],
         [-1.0000,  0.0894,  1.5938,  ..., -0.3145, -0.6094, -1.2656]]],
       dtype=torch.bfloat16, grad_fn=<ViewBackward0>)

In [11]:
from fileformer.file_dataset import ENWIK8Dataset
from torch.utils.data import DataLoader
from fileformer import ByteLevelTokenizer

In [9]:
dataset = ENWIK8Dataset('/Users/daniilogorodnikov/PycharmProjects/Notus/enwik8',
                        16828,
                        0,
                        '/Users/daniilogorodnikov/PycharmProjects/Notus/cache')

In [4]:
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [5]:
x = next(iter(dataloader))

In [8]:
x[0]

tensor([[105,  46,  34,  ...,  34, 111,  99],
        [107,  99,  34,  ..., 103, 110, 107],
        [ 34,  34,  34,  ...,  99, 110, 103],
        [116,  12,  35,  ...,  93,  69, 113]])

In [ ]:
out = model.forward(x[0], x[1].to(torch.bool))

In [ ]:
out.shape

In [12]:
tkn = ByteLevelTokenizer()

In [13]:
tkn.encode('<pad>')[0]

1

In [1]:
import sys
import math
from collections import Counter

def shannon_entropy(data: bytes) -> float:
    """
    Compute the Shannon entropy (in bits per byte) of a byte sequence.

    Parameters:
        data (bytes): Input bytes.

    Returns:
        float: Entropy in bits per byte.
    """
    if not data:
        return 0.0

    # Count frequencies of each byte (0-255)
    freq = Counter(data)
    length = len(data)

    entropy = 0.0
    for count in freq.values():
        p = count / length
        entropy -= p * math.log2(p)

    return entropy


In [2]:
with open('/Users/daniilogorodnikov/PycharmProjects/Notus/enwik8', "rb") as f:
            file_data = f.read()

In [3]:
shannon_entropy(file_data)

5.080140303348571